# [LAB 99] AutoML > 02-PyCaret(2)

## #01. 준비작업
### [1] 패키지 참조

In [10]:
from hossam import *
from pandas import concat
from pycaret.regression import *

import shap
from hossam import *



import pandas as pd
from pandas import Series, DataFrame

from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from itertools import product

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve

# 성능평가지표 모듈
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor

from pandas import merge


#BoostRegressor
from xgboost import XGBRegressor

#### 성능 평가 함수

In [3]:
def hs_get_scores(estimator, x_test, y_true):
    if hasattr(estimator, "named_steps"):
        classname = estimator.named_steps["model"].__class__.__name__
    else:
        classname = estimator.__class__.__name__

    y_pred = estimator.predict(x_test)

    return DataFrame(
        {
            "결정계수(R2)": r2_score(y_true, y_pred),
            "평균절대오차(MAE)": mean_absolute_error(y_true, y_pred),
            "평균제곱오차(MSE)": mean_squared_error(y_true, y_pred),
            "평균오차(RMSE)": np.sqrt(mean_squared_error(y_true, y_pred)),
            "평균 절대 백분오차 비율(MAPE)": mean_absolute_percentage_error(
                y_true, y_pred
            ),
            "평균 비율 오차(MPE)": np.mean((y_true - y_pred) / y_true * 100),
        },
        index=[classname],
    )

#### 변수 중요도 확인

In [4]:
from sklearn.inspection import permutation_importance

def hs_feature_importance(model, x, y):
    perm = permutation_importance(
        estimator=model,
        X=x,
        y=y,
        scoring="r2",
        n_repeats=30,
        random_state=42,
        n_jobs=-1,
    )

    # 결과 정리
    perm_df = DataFrame(
        {
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        },
        index=x.columns,
    ).sort_values("importance_mean", ascending=False)

    # 시각화
    df = perm_df.sort_values(by="importance_mean", ascending=False)

    figsize = (1280 / my_dpi, 600 / my_dpi)
    fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=my_dpi)

    sb.barplot(data=df, x="importance_mean", y=df.index)

    ax.set_title("Permutation Importance")
    ax.set_xlabel("Permutation Importance (mean)")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()

    return perm_df


#### 과적합 판정 함수

In [5]:
import matplotlib.pyplot as plt

my_dpi = 100  # 없으면 기본값

def create_figure(figsize=(1280/100, 720/100), dpi=None):
    if dpi is None:
        dpi = my_dpi
    fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi)
    return fig, ax

def finalize_plot(ax):
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()


In [6]:
def hs_learning_cv(
    estimator, x, y, scoring="neg_root_mean_squared_error",
    cv=5, train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
):

    train_sizes, train_scores, cv_scores = learning_curve(  # type: ignore
        estimator=estimator,
        X=x,
        y=y,
        train_sizes=train_sizes,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs,
        shuffle=True,
        random_state=52,
    )

    if hasattr(estimator, "named_steps"):
        classname = estimator.named_steps["model"].__class__.__name__
    else:
        classname = estimator.__class__.__name__

    # neg RMSE -> RMSE
    train_rmse = -train_scores
    cv_rmse = -cv_scores

    # 평균 / 표준편차
    train_mean = train_rmse.mean(axis=1)
    cv_mean = cv_rmse.mean(axis=1)
    cv_std = cv_rmse.std(axis=1)

    # 마지막 지점 기준 정량 판정
    final_train = train_mean[-1]
    final_cv = cv_mean[-1]
    final_std = cv_std[-1]

    gap_ratio = final_train / final_cv
    var_ratio = final_std / final_cv

    # -------------------------
    # 과소적합 기준선 (some_threshold)
    # -------------------------

    # 기준모형 RMSE (평균 예측)
    y_mean = y.mean()
    rmse_naive = np.sqrt(np.mean((y - y_mean) ** 2))

    # 분산 기반
    std_y = y.std()

    # 최소 설명력(R2) 기반
    min_r2 = 0.10
    rmse_r2 = np.sqrt((1 - min_r2) * np.var(y))

    # 최종 threshold (가장 관대한 기준)
    # → 원래 some_threshold는 도메인 지식 수준에서 이 모델은 최소 어느 정도의 성능은 내야 한다는 기준을 설정하는 것
    some_threshold = min(rmse_naive, std_y, rmse_r2)

    # -------------------------
    # 판정 로직
    # -------------------------
    if gap_ratio >= 0.95 and final_cv > some_threshold:
        status = "⚠️ 과소적합 (bias 큼)"
    elif gap_ratio <= 0.8:
        status = "⚠️ 과대적합 (variance 큼)"
    elif gap_ratio <= 0.95 and var_ratio <= 0.10:
        status = "✅ 일반화 양호"
    elif var_ratio > 0.15:
        status = "⚠️ 데이터 부족 / 분산 큼"
    else:
        status = "⚠️ 판단 유보"

    # -------------------------
    # 정량 결과 표
    # -------------------------
    result_df = DataFrame(
        {
            "Train RMSE": [final_train],
            "CV RMSE 평균": [final_cv],
            "CV RMSE 표준편차": [final_std],
            "Train/CV 비율": [gap_ratio],
            "CV 변동성 비율": [var_ratio],
            "판정 결과": [status],
        },
        index=[classname],
    )

    # display(result_df)

    # -------------------------
    # 학습곡선 시각화
    # -------------------------

    fig, ax = create_figure()

    sb.lineplot(
        x=train_sizes,
        y=train_mean,
        marker="o",
        markeredgecolor="#ffffff",
        label="Train RMSE",
    )

    sb.lineplot(
        x=train_sizes,
        y=cv_mean,
        marker="o",
        markeredgecolor="#ffffff",
        label="Train RMSE",
    )

    ax.set_xlabel("RMSE", fontsize=8, labelpad=5)  # type: ignore
    ax.set_ylabel("학습곡선 (Learning Curve)", fontsize=8, labelpad=5)  # type: ignore
    ax.grid(True, alpha=0.3)  # type: ignore

    finalize_plot(ax)

    return result_df


#### 성능평가 + 과적합 판정 동시 수행 함수

In [7]:
def hs_get_score_cv(
    estimator,
    x_test,
    y_test,
    x_origin,
    y_origin,
    scoring="neg_root_mean_squared_error",
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
):

    score_df = hs_get_scores(estimator, x_test, y_test)

    cv_df = hs_learning_cv(
        estimator,
        x_origin,
        y_origin,
        scoring=scoring,
        cv=cv,
        train_sizes=train_sizes,
        n_jobs=n_jobs,
    )

    return merge(score_df, cv_df, left_index=True, right_index=True)


#### SHAP 분석

In [8]:
def hs_shap_analysis(
    model,
    x: DataFrame,
    plot: bool = True,
    width: int = 1600,
    height: int = 800,
):
    # -------------------------------------------------
    # 1. SHAP Explainer 생성 (트리 모델용)
    # -------------------------------------------------
    explainer = shap.TreeExplainer(model)

    # -------------------------------------------------
    # 2. SHAP 값 계산
    # shape = (n_samples, n_features)
    # -------------------------------------------------
    shap_values = explainer.shap_values(x)

    # -------------------------------------------------
    # 3. SHAP DataFrame 변환
    # -------------------------------------------------
    shap_df = DataFrame(
        shap_values,
        columns=x.columns,
        index=x.index,
    )

    # -------------------------------------------------
    # 4. SHAP 요약 통계
    # -------------------------------------------------
    summary_df = DataFrame(
        {
            "feature": shap_df.columns,
            "mean_abs_shap": shap_df.abs().mean().values,
            "mean_shap": shap_df.mean().values,
            "std_shap": shap_df.std().values,
        }
    )

    # -------------------------------------------------
    # 5. 영향 방향 (보수적 기준)
    # -------------------------------------------------
    summary_df["direction"] = np.where(
        summary_df["mean_shap"] > 0,
        "양(+) 경향",
        np.where(summary_df["mean_shap"] < 0, "음(-) 경향", "혼합/미약"),
    )

    # -------------------------------------------------
    # 6. 변동성 지표 (안정성 판단)
    # -------------------------------------------------
    summary_df["cv"] = (
        summary_df["std_shap"] / (summary_df["mean_abs_shap"] + 1e-9)
    )

    summary_df["variability"] = np.where(
        summary_df["cv"] < 1,
        "stable",      # 평균 대비 변동성 낮음
        "variable",    # 상황 의존적 영향
    )

    # -------------------------------------------------
    # 7. 중요도 기준 정렬
    # -------------------------------------------------
    summary_df = (
        summary_df
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    # -------------------------------------------------
    # 8. 중요 변수 판별 (누적 80%)
    # -------------------------------------------------
    total_importance = summary_df["mean_abs_shap"].sum()

    summary_df["importance_ratio"] = (
        summary_df["mean_abs_shap"] / total_importance
    )

    summary_df["importance_cumsum"] = (
        summary_df["importance_ratio"].cumsum()
    )

    summary_df["is_important"] = np.where(
        summary_df["importance_cumsum"] <= 0.80,
        "core",        # 핵심 변수
        "secondary",   # 보조 변수
    )

    # -------------------------------------------------
    # 9. SHAP Summary Plot
    # -------------------------------------------------
    if plot:
        shap.summary_plot(shap_values, x, show=False)

        fig = plt.gcf()
        fig.set_size_inches(width / 100, height / 100)

        plt.title("SHAP Summary Plot", fontsize=10, pad=10)
        plt.xlabel("SHAP value", fontsize=8)
        plt.xticks(fontsize=6)
        plt.yticks(fontsize=8)
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        plt.close()

    # -------------------------------------------------
    return summary_df, shap_values


#### SHAP dependance

In [9]:
def hs_shap_dependence_analysis(
    summary_df: DataFrame,
    shap_values,
    x_train: DataFrame,
    include_secondary: bool = False, #core 변수가 너무 적을 때, True 로 설정하면 secondary 까지 포함
    width: int = 1600,
    height: int = 800,
):
    # 1. 주 대상 변수 (Core + Variable)
    main_features = summary_df[
        (summary_df["is_important"] == "core")
        & (summary_df["variability"] == "variable")
    ]["feature"].tolist()

    # 2. 상호작용 후보 변수
    interaction_features = summary_df[
        summary_df["is_important"] == "core"
    ]["feature"].tolist()

    if include_secondary and len(interaction_features) < 2:
        interaction_features.extend(
            summary_df[summary_df["is_important"] == "secondary"]["feature"].tolist()
        )

    # 3. 변수 쌍 생성 (자기 자신 제외)
    pairs = []
    for f in main_features:
        for inter in interaction_features:
            # 자기 자신과의 조합은 제외
            if f != inter:
                pairs.append((f, inter))

    # 중요도 순 정렬 (주 변수 기준)
    importance_rank = {}
    for i, row in summary_df.iterrows():
        importance_rank[row["feature"]] = i

    pairs = sorted(
        pairs,
        key=lambda x: importance_rank.get(x[0], 999),
    )

    # 4. dependence plot 일괄 생성
    for feature_name, interaction_name in pairs:
        shap.dependence_plot(
            feature_name,
            shap_values,
            x_train,
            interaction_index=interaction_name,
            show=False,
        )

        # SHAP figure 직접 제어
        fig = plt.gcf()
        fig.set_size_inches(width / my_dpi, height / my_dpi)

        plt.title(
            f"SHAP Dependence Plot: {feature_name} × {interaction_name}",
            fontsize=10,
            pad=10,
        )

        plt.xlabel(feature_name, fontsize=10)
        plt.ylabel(f"SHAP value for {feature_name}", fontsize=10)

        plt.xticks(fontsize=6)
        plt.yticks(fontsize=8)
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        plt.close()

    return pairs


### [2] 데이터 불러오기 + 인덱스 설정 + 카테고리 타입 지정

In [11]:
origin = load_data("restaurant_sales_preprocessed")
origin.set_index("date", inplace=True)


# XGBoost 라서 astype int 처리
origin["holiday"] = origin["holiday"].astype("int")
origin["weekend"] = origin["weekend"].astype("int")
origin.info()


어느 식당의 1년간 일별 매출을 기록한 데이터의 전처리 완료 버전(명목형이 이진변수만 있으므로 더미변수는 처리하지 않음)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 353 entries, 2024-01-01 to 2024-12-30
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales           353 non-null    float64
 1   visitors        353 non-null    int64  
 2   avg_price       353 non-null    int64  
 3   marketing_cost  353 non-null    float64
 4   delivery_ratio  353 non-null    float64
 5   rain_mm         353 non-null    float64
 6   temperature     353 non-null    float64
 7   holiday         353 non-null    int32  
 8   weekend         353 non-null    int32  
dtypes: float64(5), int32(2), int64(2)
memory usage: 24.8 KB


## #02. PyCaret setup

In [12]:
from pycaret.regression import RegressionExperiment

s = RegressionExperiment()

s.setup(
    # -------------------------------
    # 데이터셋 지정
    # -------------------------------
    data=origin,
    
    # 예측 목표 변수 지정
    target="sales",
    
    # 랜덤 시드 고정
    session_id=52,
    
    # 훈련 데이터 비율 (기본값=0.7)
    train_size=0.75,
    
    # 교차검증 폴드 수 지정
    fold=5,
    
    # 최적화 출력 완화 (기본값 True, False 권장)
    verbose=False,
    
    # GPU 사용 여부
    # use_gpu=True,
    
    # -------------------------------
    # 전처리 설정
    # -------------------------------
    
    # 범주형 변수 지정 (기본값 None)
    categorical_features=['weekend', 'holiday'],
    
    # 생략할 변수 지정 (기본값 None)
    ignore_features=[],
    
    # 데이터 정규화 / 표준화 활성화 (기본값 False)
    normalize=True,
    
    # 데이터 정규화 / 표준화 방법 선택
    # 'minmax', 'maxabs', 'robust', 'zscore'
    normalize_method='zscore',
    
    # --------------------------------
    # 아래 기능은 사용하지 않음
    # (데이터 별도 전처리 단계를 거치는 것을 권장)
    # --------------------------------
    
    # 이상치 제거 (기본값 False, IQR 범위 밖 값 삭제)
    remove_outliers=False,
    
    # 종속변수 변환
    transform_target=False,
    
    # 변수 선택 (기본값 False)
    feature_selection=False
)


In [13]:
s.pull()

,Description,Value
0,Session id,52
1,Target,sales
2,Target type,Regression
3,Original data shape,"(353, 9)"
4,Transformed data shape,"(353, 9)"
5,Transformed train set shape,"(264, 9)"
6,Transformed test set shape,"(89, 9)"
7,Numeric features,6
8,Categorical features,2
9,Preprocess,True


## #03. 베이스 모델 구축하기
### [1] 사용할 수 있는 학습 모델의 종류 확인

In [14]:
s.models()

,Name,Reference,Turbo
ID,,,
lr,Linear Regression,sklearn.linear_model._base.LinearRegression,True
lasso,Lasso Regression,sklearn.linear_model._coordinate_descent.Lasso,True
ridge,Ridge Regression,sklearn.linear_model._ridge.Ridge,True
en,Elastic Net,sklearn.linear_model._coordinate_descent.ElasticNet,True
lar,Least Angle Regression,sklearn.linear_model._least_angle.Lars,True
llar,Lasso Least Angle Regression,sklearn.linear_model._least_angle.LassoLars,True
omp,Orthogonal Matching Pursuit,sklearn.linear_model._omp.OrthogonalMatchingPursuit,True
br,Bayesian Ridge,sklearn.linear_model._bayes.BayesianRidge,True
ard,Automatic Relevance Determination,sklearn.linear_model._bayes.ARDRegression,False


### [2] 베이스 모델 성능 비교
#### 모든 모델에 대한 성능 비교

In [15]:
best5models = s.compare_models(sort='RMSE',n_select=5,fold=5)
best5models

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
br,Bayesian Ridge,0.1654,0.0427,0.2056,0.6959,0.0119,0.0101,0.0120
ridge,Ridge Regression,0.1657,0.0428,0.2060,0.6948,0.0119,0.0101,0.4060
lr,Linear Regression,0.1658,0.0429,0.2061,0.6943,0.0119,0.0101,0.5880
lar,Least Angle Regression,0.1658,0.0429,0.2061,0.6943,0.0119,0.0101,0.0120
huber,Huber Regressor,0.1653,0.0429,0.2062,0.6938,0.0119,0.0101,0.0140
rf,Random Forest Regressor,0.1751,0.0494,0.2214,0.6438,0.0128,0.0107,0.0360
ada,AdaBoost Regressor,0.1758,0.0500,0.2227,0.6430,0.0128,0.0108,0.0260
et,Extra Trees Regressor,0.1783,0.0502,0.2231,0.6391,0.0129,0.0109,0.0300
lightgbm,Light Gradient Boosting Machine,0.1799,0.0517,0.2256,0.6329,0.0130,0.0110,0.0320
catboost,CatBoost Regressor,0.1851,0.0534,0.2299,0.6235,0.0132,0.0113,0.3760


#### 주요 모델만 선정하여 성능 비교
수업에서 사용한 모델만을 대상으로 베이스모델간의 성능 비교

In [16]:
best5models = s.compare_models(
    include=[
        "lr", "ridge", "lasso", "en",
        "knn", "svm",
        "dt", "xgboost", "lightgbm", "catboost"
    ],
    sort="RMSE",
    n_select=5,
    fold=5
)

best5models


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.1657,0.0428,0.2060,0.6948,0.0119,0.0101,0.4480
lr,Linear Regression,0.1658,0.0429,0.2061,0.6943,0.0119,0.0101,0.0800
lightgbm,Light Gradient Boosting Machine,0.1799,0.0517,0.2256,0.6329,0.0130,0.0110,0.0320
svm,Support Vector Regression,0.1811,0.0526,0.2284,0.6323,0.0131,0.0111,0.0120
catboost,CatBoost Regressor,0.1851,0.0534,0.2299,0.6235,0.0132,0.0113,0.3720
xgboost,Extreme Gradient Boosting,0.1884,0.0552,0.2344,0.6114,0.0135,0.0115,0.0220
knn,K Neighbors Regressor,0.1878,0.0558,0.2358,0.6041,0.0136,0.0115,0.0180
dt,Decision Tree Regressor,0.2458,0.0994,0.3135,0.2846,0.0181,0.0150,0.0180
lasso,Lasso Regression,0.3208,0.1470,0.3824,-0.0212,0.0220,0.0196,0.4440
en,Elastic Net,0.3208,0.1470,0.3824,-0.0212,0.0220,0.0196,0.3980


[Ridge(random_state=52),
 LinearRegression(n_jobs=-1),
 LGBMRegressor(n_jobs=-1, random_state=52),
 SVR(),

## #03. 앙상블
#### [1] 선정된 모형에 대한 Voting

In [17]:
blended = s.blend_models(
    estimator_list=best5models,
    fold=5
)

blended


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1922,0.0578,0.2404,0.6162,0.0138,0.0117
1,0.1795,0.0475,0.2180,0.7314,0.0126,0.0110
2,0.1467,0.0330,0.1817,0.7925,0.0106,0.0090
3,0.1535,0.0364,0.1908,0.6840,0.0110,0.0094
4,0.1757,0.0494,0.2223,0.5888,0.0128,0.0107
Mean,0.1695,0.0448,0.2106,0.6826,0.0121,0.0104
Std,0.0169,0.0090,0.0215,0.0744,0.0012,0.0010


VotingRegressor(estimators=[('Ridge Regression', Ridge(random_state=52)),
                            ('Linear Regression', LinearRegression(n_jobs=-1)),
                            ('Light Gradient Boosting Machine',
                             LGBMRegressor(n_jobs=-1, random_state=52)),
                            ('Support Vector Regression', SVR()),
                            ('CatBoost Regressor',
                             <catboost.core.CatBoostRegressor object at 0x000001FACC0EE790>)],
                n_jobs=-1)

In [ ]:
%%time
tuned = s.tune_model(
    estimator=blended,
    optimize="RMSE",
    n_iter=30,
    fold=5,
    choose_better=True,
    verbose=False,
    early_stopping=True,

    # 하이퍼파라미터 탐색 방법
    # grid = 전체탐색, random = 무작위탐색
    search_algorithm="grid",

    # 직접 하이퍼파라미터 범위 구성 (생략할 경우 자동 구성)
    custom_grid={
        # Ridge Regression
        "Ridge Regression__alpha": [0.01, 0.1, 1, 10, 100],

        # Light Gradient Boosting Machine
        "Light Gradient Boosting Machine__n_estimators": [200],
        "Light Gradient Boosting Machine__learning_rate": [0.05, 0.1],
        "Light Gradient Boosting Machine__num_leaves": [31, 63],
        "Light Gradient Boosting Machine__max_depth": [-1, 5],
        "Light Gradient Boosting Machine__min_child_samples": [20, 50],
        "Light Gradient Boosting Machine__subsample": [0.8],
        "Light Gradient Boosting Machine__reg_alpha": [0, 0.1, 1],
        "Light Gradient Boosting Machine__reg_lambda": [0, 1, 5],

        # Support Vector Regression
        "Support Vector Regression__kernel": ["rbf"],  # 수업·실습 기준 rbf 고정
        "Support Vector Regression__C": [0.1, 1, 10, 100],
        "Support Vector Regression__epsilon": [0.01, 0.05, 0.1, 0.2],
        "Support Vector Regression__gamma": ["scale", "auto", 0.01, 0.1, 1],

        # CatBoost Regressor
        "CatBoost Regressor__iterations": [300, 500],
        "CatBoost Regressor__learning_rate": [0.01, 0.03, 0.1],
        "CatBoost Regressor__depth": [4, 6, 8],
        "CatBoost Regressor__l2_leaf_reg": [1, 3, 5],
        "CatBoost Regressor__subsample": [0.8, 1.0],
    }
)

tuned


CPU times: total: 3min 58s
Wall time: 17h 10min 31s


### [2] 하이퍼 파라미터 확인


In [ ]:
tuned.get_params()

## #04. 분석 결과 확인하기
### [1] 훈련, 검증 데이터 추출
- 전처리가 완료된 데이터를 반환한다
- get_config() 는 pycaret.regressor 객체의 메서드

#### 각 데이터 추출

In [ ]:
# 원본 독립변수 (분할 전)
X = s.get_config("X")
y = s.get_config("y")

# 훈련 데이터 (데이터 변환 전)
X_train = s.get_config("X_train")
y_train = s.get_config("y_train")

# 테스트 데이터 (데이터 변환 전)
X_test = s.get_config("X_test")
y_test = s.get_config("y_test")

# 변환된 훈련 데이터
X_train_transformed = s.get_config("X_train_transformed")
y_train_transformed = s.get_config("y_train_transformed")

# 변환된 테스트 데이터
X_test_transformed = s.get_config("X_test_transformed")
y_test_transformed = s.get_config("y_test_transformed")

# 변환된 훈련 + 테스트 데이터셋
X_transformed = concat([X_train_transformed, X_test_transformed])
y_transformed = concat([y_train_transformed, y_test_transformed])

# 데이터 크기 확인
(
    X.shape, y.shape,
    (X_train.shape, y_train.shape),
    (X_test.shape, y_test.shape),
    (X_train_transformed.shape, y_train_transformed.shape),
    (X_test_transformed.shape, y_test_transformed.shape)
)


### [2] 성능평가
#### 시각화 초기화

In [ ]:
init_pyplot()

#### 성능평가 지표 확인

In [ ]:
%%time
hs_get_score_cv(
    tuned,
    X_train_transformed,
    y_train_transformed,
    X_transformed,
    y_transformed
)


### [3] 변수 중요도 확인

In [ ]:
hs_feature_importance(tuned,X_train_transformed,y_train_transformed)